In [1]:


#@title The MIT License (MIT)
#
# Copyright (c) 2026 Eric dos Santos.
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in
# all copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
# THE SOFTWARE.

# Fake News Classification

This project aims to develop a neural network for detecting fake news in Portuguese, using the dataset [Fake.br-Corpus](https://github.com/roneysco/Fake.br-Corpus). With this, we seek to create a system capable of identifying patterns and distinguishing fake news from real news, contributing to the fight against misinformation.

<table class="tfo-notebook-buttons" align="center">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/ericshantos/br_fake_news_detector/blob/main/br_fake_news_detector_model.ipynb
"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run on Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/ericshantos/br_fake_news_detector/blob/main/br_fake_news_detector_model.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View the code on GitHub</a>
  </td>
</table>

## Dataset loading

In [2]:
from pathlib import Path
import pandas as pd
import os

In [3]:
!git clone https://github.com/roneysco/Fake.br-Corpus
DATA_PATH = Path("./Fake.br-Corpus/full_texts")

Cloning into 'Fake.br-Corpus'...
remote: Enumerating objects: 28763, done.
remote: Total 28763 (delta 0), reused 0 (delta 0), pack-reused 28763 (from 1)
Receiving objects: 100% (28763/28763), 37.10 MiB | 3.00 MiB/s, done.
Resolving deltas: 100% (14129/14129), done.
Updating files: 100% (21602/21602), done.


In [4]:
# News Directory
fake_dir = DATA_PATH / "fake"
real_dir = DATA_PATH / "true"

### News content extraction:


In [5]:
import os
import pandas as pd

def load_news(news_dir: str, label: str) -> pd.DataFrame:
    # List to store news
    news = []

    # Cycle through all files in the specified directory
    for filename in os.listdir(news_dir):
        # Checks if the file has the .txt extension
        if filename.endswith(".txt"):
            # Gets the full path of the file
            file_path = os.path.join(news_dir, filename)

            # Open the file and read its contents
            with open(file_path, "r") as file:
                content = file.read()

                # Adds the content and label to the news list
                news.append({"text": content, "label": label})

    # Returns a pandas DataFrame containing the news
    return pd.DataFrame(news)

Result:

In [6]:
fake_news = load_news(fake_dir, 0)
real_news = load_news(real_dir, 1)

## Data preprocessing

In [7]:
from torch.utils.data import Dataset, DataLoader

ModuleNotFoundError: No module named 'torch'

### Concatenate the DataFrames

Group Dataframes to generate a single robust database.

In [ ]:
data_news = pd.concat(
  [fake_news, real_news], ignore_index=True
).sample(frac=1, random_state=13)

Final base information:

In [ ]:
data_news.info()

In [ ]:
data_news = data_news.apply(

    # If valid, type the column as float
    lambda col: col.astype(float) if col.apply(

        # Check if they are digits
        lambda x: str(x).replace('.', '', 1).isdigit()
    ).all() else col
)

# Result
print(data_news.dtypes)

### Data cleaning

In [ ]:
!python -m spacy download pt_core_news_sm > /dev/null 2>&1
!pip install unidecode > /dev/null 2>&1

from unidecode import unidecode
import spacy
from tqdm import tqdm
import torch

In [ ]:
def clean_text_batch(texts, batch_size=100):
    nlp = spacy.load("pt_core_news_sm")
    cleaned_texts = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Limpando textos"):
        batch = texts[i:i+batch_size]
        batch_cleaned = []

        for text in batch:
            doc = nlp(text)
            tokens = [unidecode(token.lemma_) for token in doc
                     if not token.is_stop and not token.is_punct]
            batch_cleaned.append(' '.join(tokens))

        cleaned_texts.extend(batch_cleaned)

        del batch
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return cleaned_texts

Clear news content:

In [ ]:
data_news["text"] = clean_text_batch(data_news["text"].tolist())

## Training

In [ ]:
from transformers import AutoTokenizer

### Splitting the dataset into training and testing

In [ ]:
from sklearn.model_selection import train_test_split

# Splits data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
  data_news['text'],
  data_news['label'],
  test_size=0.2,
  random_state=13,
  stratify=data_news['label']
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

### Create tokenizer

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
  "neuralmind/bert-base-portuguese-cased"
)

## Architectures

In [ ]:
class NewsDataset(Dataset):
  def __init__(self, texts, label, tokenizer, max_length=256):
    self.texts = texts.tolist() if hasattr(texts, 'tolist') else texts
    self.label = label.tolist() if hasattr(label, 'tolist') else label
    self.tokenizer = tokenizer

    self.max_length = max_length

  def __getitem__(self, idx: int):
    encoding = self.tokenizer(
        self.texts[idx],
        padding='max_length',
        truncation=True,
        max_length=self.max_length,
        return_tensors="pt"
    )

    return {
        "input_ids": encoding["input_ids"].squeeze(0),
        "attention_mask": encoding["attention_mask"].squeeze(0),
        "label": torch.tensor(self.label[idx], dtype=torch.long)
    }

  def __len__(self) -> int:
    return len(self.texts)

In [ ]:
dataset_train = NewsDataset(X_train, y_train, tokenizer)
dataset_test = NewsDataset(X_test, y_test, tokenizer)

In [ ]:
loader_train = DataLoader(
  dataset_train,
  batch_size=8,
  shuffle=True,
  pin_memory=True,
  num_workers=2,
  prefetch_factor=2
)

loader_test = DataLoader(
  dataset_test,
  batch_size=8,
  pin_memory=True,
  num_workers=2
)

### Model architecture

In [ ]:
import torch.nn as nn
from transformers import AutoModel

In [ ]:
class BERTClassifier(nn.Module):

  def __init__(self, drop_rate: int = 0.2) -> None:
    super().__init__()

    self.bert = AutoModel.from_pretrained(
      "neuralmind/bert-base-portuguese-cased"
    )

    for param in self.bert.embeddings.parameters():
      param.requires_grad = False

    hidden = self.bert.config.hidden_size

    self.classifier = nn.Sequential(
        nn.Linear(hidden, 64),
        nn.GELU(),
        nn.Dropout(drop_rate),

        nn.Linear(64, 32),
        nn.GELU(),
        nn.Dropout(drop_rate),

        nn.Linear(32, 1),
    )

  def forward(self, input_ids, attention_mask):
    outputs = self.bert(
      input_ids=input_ids,
      attention_mask=attention_mask
    )

    pooler = outputs.last_hidden_state[:, 0, :]
    return self.classifier(pooler)

  def save(self, path: str):
    """Salva o modelo e o tokenizer"""
    # Salva os pesos do modelo
    torch.save({
      'model_state_dict': self.state_dict(),
      'bert_config': self.bert.config
    }, f"{path}/model_weights.pth")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = BERTClassifier().to(device)

**Model compilation**:

In [ ]:
import torch.optim as optim

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

### Training the model

In [ ]:
epochs = 5

In [ ]:
epochs = 5
accumulation_steps = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    progress_bar = tqdm(loader_train, desc=f"Epoch {epoch+1}")

    for batch_idx, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask).squeeze()
        loss = criterion(outputs, labels.float())

        loss = loss / accumulation_steps
        loss.backward()

        total_loss += loss.item() * accumulation_steps

        if (batch_idx + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        progress_bar.set_postfix({
            "batch_loss": f"{loss.item() * accumulation_steps:.4f}"
        })

        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()

    if (batch_idx + 1) % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = total_loss / len(loader_train)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")

#### Model evaluation

In [ ]:
model.eval()

correct = 0
total = 0
predictions = []
all_labels = []

with torch.no_grad():
    for batch in loader_test:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.sigmoid(outputs.squeeze())
        preds = (probs > 0.5).long()

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        predictions.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total
print(f"Acurácia: {accuracy:.4f}")

## Save to model

In [ ]:
torch.save({
  "model_state_dict": model.state_dict(),
  "bert_config": model.bert.config,
  },
  "veritas-bert-ptbr.pth"
)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoConfig

model_name = "neuralmind/bert-base-portuguese-cased"

config = AutoConfig.from_pretrained(model_name, num_labels=2)

model = AutoModelForSequenceClassification.from_config(config)

checkpoint = torch.load("veritas-bert-ptbr.pth", map_location="cpu")
model.load_state_dict(checkpoint["model_state_dict"])

model.save_pretrained("veritas_bert/")